# Chat Completions API

In [3]:
# 공통 설정
# .env 파일에서 API Key와 기본 모델 명을 읽어온다
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY를 환경변수로 설정하세요")

client = OpenAI(api_key=api_key)
DEFAULT_MODEL = os.getenv("OPENAI_DEFAULT_MODEL","gpt-4.1-mini")

print("OpenAI client 준비 완료")
print("기본 모델 :", DEFAULT_MODEL)

OpenAI client 준비 완료
기본 모델 : gpt-4.1-mini


## 기본 호출
- system 메세지는 모델의 역할과 답변 규칙을 지정한다.
- user 메세지는 사용자의 실제 요청이다.

In [4]:
response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[
        {"role":"system","content":"너는 초급 개발자에게 쉽게 설명하는 AI 강사이다."},
        {"role":"user", "content":"Chat Completions API의 messages 구조를 설명해줘."}
    ],
    temperature=0.3
)

print(response.choices[0].message.content)

물론이야! Chat Completions API에서 사용하는 **messages** 구조는 대화 내용을 컴퓨터가 이해할 수 있도록 정리한 거야. 쉽게 말해서, 대화의 각 한 줄을 객체(object)로 표현한 리스트(list)라고 생각하면 돼.

### messages 구조 기본 형태

```json
[
  {
    "role": "사용자 역할",
    "content": "메시지 내용"
  },
  {
    "role": "시스템 역할",
    "content": "메시지 내용"
  },
  {
    "role": "어시스턴트 역할",
    "content": "메시지 내용"
  }
]
```

### 각 필드 설명

- **role**: 누가 말했는지를 나타내는 역할(role)이야. 보통 3가지가 있어.
  - `"system"`: 대화의 규칙이나 설정을 알려주는 역할. 예를 들어, "친절하게 답해줘" 같은 지시를 줄 때 사용해.
  - `"user"`: 실제 사용자가 보낸 메시지.
  - `"assistant"`: AI가 답변한 메시지.

- **content**: 실제 대화 내용(텍스트)이 들어가는 부분이야.

### 예시

```json
[
  {
    "role": "system",
    "content": "너는 친절한 도우미야."
  },
  {
    "role": "user",
    "content": "안녕! 오늘 날씨 어때?"
  },
  {
    "role": "assistant",
    "content": "안녕하세요! 오늘은 맑고 따뜻해요."
  }
]
```

### 요약

- messages는 대화의 흐름을 역할별로 나눈 배열(리스트)야.
- 각 메시지는 누가 말했는지(role)와 무슨 말을 했는지(content)를 포함해.
- 시스템 메시지는 AI에게 지시를 줄 때 쓰고, 사용자 메시지는 질문이나 요청, 어시스턴트 메시지는 AI의 답변을 담아.

이해하기 쉽도록 설명했는데, 더 궁금한 점 있으면 언제든 물어봐!

## 대화 이력 직접 누적하기
Chat completions API에서는 이전 대화를 API가 자동으로 기억하지 않는다.
따라서 이어지는 대화를 만드려면 message 리스트에 사용자 질문과 모델 답변을 직접 누적해야 한다.

In [5]:
messages=[
    {"role":"system","content":"너는 Python 수업을 돕는 AI 튜터이다. 답변은 3문장 이내로 한다."}
]

def chat(user_input):
    # 사용자의 새 질문을 대화 이력에 추가
    messages.append({"role":"user","content":user_input})

    # 지금까지의 전체 대화 이력을 모델에 전달
    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=messages,
        temperature=0.4
    )

    # 모델 답변을 꺼내고 다음 턴을 위해 다시 이력에 추가
    answer = response.choices[0].message.content
    messages.append({"role":"assistant","content":answer})
    return answer

print(chat("함수와 메서드의 차이를 설명해줘."))
print()
print(chat("방금 설명을 Java 예시로 바꿔줘."))

함수는 독립적으로 정의되어 호출되는 코드 블록이고, 메서드는 특정 객체에 속한 함수입니다. 메서드는 객체의 상태를 다루거나 객체와 관련된 동작을 수행합니다. 즉, 메서드는 클래스 내부에 정의된 함수라고 볼 수 있습니다.

Java에서 함수는 클래스 외부에 독립적으로 존재하지 않고, 모든 함수는 메서드로서 클래스 내부에 정의됩니다. 예를 들어, `public static void print()`는 클래스에 속한 메서드이고, 객체를 통해 호출할 수 있습니다. 따라서 Java에서는 메서드가 함수의 역할을 모두 수행합니다.


## system 메세지 변경을 통해 답변 스타일 변경

In [6]:
system_message = [
    "너는 초급 개발자 대상 강사다. 쉬운 용어와 간단한 예시로 설명한다.",
    "너는 백엔드 실무자에게 설명하는 멘토다. 실무 예시를 포함해서 설명한다.",
    "너는 백엔드 개발자 면접관이다. 핵심 개념과 실무 판단 기준을 중심으로 설명한다."
]

user_question = "API 토큰 사용량을 로깅해야 하는 이유를 설명해줘"

for idx,system_message in enumerate(system_message,start=1):
    pratice_messages = [
        {"role":"system","content":system_message},
        {"role":"user","content":user_question}
    ]

    response = client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=pratice_messages,
        temperature=0.3
    )

    print(f"[{idx}] {system_message}")
    print(response.choices[0].message.content)
    print("-" * 60)

[1] 너는 초급 개발자 대상 강사다. 쉬운 용어와 간단한 예시로 설명한다.
좋아! API 토큰 사용량을 로깅해야 하는 이유를 쉽게 설명할게.

1. **누가 얼마나 썼는지 알기 위해서**  
   API 토큰은 어떤 사람이 API를 사용할 때 주는 '열쇠' 같은 거야. 누가 몇 번이나 썼는지 기록해두면, 누가 많이 썼는지 알 수 있어.

2. **문제가 생겼을 때 원인 찾기 위해서**  
   만약 API가 갑자기 느려지거나 오류가 나면, 누가 언제 많이 썼는지 기록이 있으면 문제를 찾기 쉬워.

3. **남용 방지**  
   어떤 사람이 너무 많이 API를 쓰면 서버에 부담이 될 수 있어. 사용량을 기록해서 너무 많이 쓰면 알려주거나 막을 수 있어.

4. **비용 관리**  
   API 사용량에 따라 비용이 발생할 수도 있어. 누가 얼마나 썼는지 기록하면 비용을 정확히 계산할 수 있어.

예를 들어, 카페에서 음료를 주문할 때 쿠폰을 써야 한다고 생각해봐. 쿠폰을 몇 번 썼는지 기록해두면, 누가 쿠폰을 많이 썼는지 알 수 있고, 문제 생기면 확인할 수 있지.

이해됐지? 더 궁금한 거 있으면 물어봐!
------------------------------------------------------------
[2] 너는 백엔드 실무자에게 설명하는 멘토다. 실무 예시를 포함해서 설명한다.
API 토큰 사용량을 로깅하는 이유는 여러 가지가 있는데, 실무에서 특히 중요한 점들을 중심으로 설명할게.

### 1. 보안 감시 및 이상 탐지
- **이유:** API 토큰이 유출되거나 악용될 경우, 비정상적인 사용 패턴이 나타날 수 있어. 이를 조기에 감지하려면 사용량 로그가 필수적이야.
- **실무 예시:**  
  어떤 서비스에서 갑자기 특정 API 토큰으로 평소보다 10배 이상 요청이 들어왔다고 가정해보자. 로그를 통해 이 비정상적인 트래픽을 감지하고 해당 토큰을 즉시 비활성화하거나 재발급할 수 있어.

### 2. 비용 관리 및 과금
- **이유:** 클라